# 05 — Hero Image Generator

**Goal:** Prove that one or more worn photos of a garment can be turned into a clean studio flat-lay
"hero" image suitable for a closet card — without ever staging a flat-lay shoot.

**Pipeline:**
```
1–N worn / flat photos
  → Step A  baseline cutouts (rembg / BiRefNet)
  → Step B  single-image hero  (Gemini image-conditioned, photo[0] only)
  → Step C  multi-image hero   (Gemini, all N photos)
  → Step D  incremental refinement  (add one photo at a time, see each delta)
  → Step E  two-stage variant  (rembg cutout as structural prior + original)
  → Step F  metrics table
  → Decision note
```

All API calls go through `lib/hero_image.generate_hero()` — see `lib/hero_image.py` for the prompt.

In [ ]:
import os, sys, glob
from pathlib import Path

import matplotlib.pyplot as plt
from dotenv import load_dotenv
from PIL import Image

load_dotenv(Path('..') / '.env')
sys.path.insert(0, str(Path('..')))

from lib.hero_image import (
    generate_hero,
    IMAGE_GEN_MODEL,
    IMAGE_GEN_MODEL_V2,
    IMAGE_GEN_MODEL_LITE,
)
from lib.images import resize_image
from lib.metrics import HeroMetricsRecorder, compute_white_pixel_ratio, compute_color_delta_lab
from lib.remove_background import remove_background, REMBG_AVAILABLE

OUT_DIR = Path('../outputs/05')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'IMAGE_GEN_MODEL      (primary):  {IMAGE_GEN_MODEL}')
print(f'IMAGE_GEN_MODEL_V2   (older):    {IMAGE_GEN_MODEL_V2}')
print(f'IMAGE_GEN_MODEL_LITE (new lite): {IMAGE_GEN_MODEL_LITE}')
print(f'API key set: {bool(os.getenv("GEMINI_API_KEY"))}')
print(f'rembg available: {REMBG_AVAILABLE}')
print(f'FAL_KEY set: {bool(os.getenv("FAL_KEY"))}')

## Step 1 — Load 1–N photos of one item

Priority order:
1. `inputs/worn/` — add your mirror selfies / outfit photos here (see README in that folder)
2. `inputs/multi_angle/` — same-item multi-angle set from notebook 03
3. First 2 DB items in `~/.closet/images/items/` as a last-resort placeholder

The notebook uses `imgs[0]` as the "single worn photo" baseline (Step B) and all of `imgs` for the multi-photo runs (Steps C–E).

In [ ]:
WORN_DIR = Path('../inputs/worn')
MULTI_ANGLE_DIR = Path('../inputs/multi_angle')

def _load_from(directory: Path) -> list[Path]:
    paths = sorted(directory.glob('*.jpg')) + sorted(directory.glob('*.jpeg')) + sorted(directory.glob('*.png'))
    return [p for p in paths if not p.name.startswith('.') and p.suffix.lower() != '.md']

if worn_paths := _load_from(WORN_DIR):
    PHOTO_PATHS = worn_paths
    source = 'inputs/worn/'
elif MULTI_ANGLE_DIR.exists() and (ma_paths := _load_from(MULTI_ANGLE_DIR)):
    PHOTO_PATHS = ma_paths
    source = 'inputs/multi_angle/'
else:
    db_items = sorted(glob.glob(os.path.expanduser('~/.closet/images/items/*.jpg')))
    PHOTO_PATHS = [Path(p) for p in db_items[:2]]
    source = '~/.closet/images/items/ (first 2 — add real worn photos for a meaningful test)'

print(f'Source: {source}')
print(f'Loaded {len(PHOTO_PATHS)} photo(s):')
for p in PHOTO_PATHS:
    print(f'  {p}')

imgs = [resize_image(Image.open(p)) for p in PHOTO_PATHS]

cols = max(len(imgs), 1)
fig, axes = plt.subplots(1, cols, figsize=(cols * 4, 5))
if cols == 1:
    axes = [axes]
for ax, img, p in zip(axes, imgs, PHOTO_PATHS):
    ax.imshow(img)
    ax.set_title(Path(p).name, fontsize=8)
    ax.axis('off')
plt.suptitle('Input photos', fontsize=12)
plt.tight_layout()
plt.show()

## Step A — Baseline cutouts

**What:** Remove the background with rembg (local) and optionally BiRefNet via fal.ai.

**Why:** Establishes what naive background-removal looks like without any generation.
The cutout is NOT the hero image — it will have the person removed but still be a
worn-garment crop, not a flat-lay. This is the control condition.

Set `FAL_KEY` in `.env` to also run the BiRefNet comparison.

In [ ]:
cutouts = {}

if REMBG_AVAILABLE:
    print('Running rembg on photo[0]...')
    try:
        cutouts['rembg'] = remove_background(imgs[0], provider='rembg')
        print('  done.')
    except Exception as e:
        print(f'  rembg failed: {e}')
else:
    print('rembg not installed — skipping (pip install rembg onnxruntime)')

if os.getenv('FAL_KEY'):
    print('Running BiRefNet (fal.ai) on photo[0]...')
    try:
        cutouts['birefnet'] = remove_background(imgs[0], provider='fal')
        print('  done.')
    except Exception as e:
        print(f'  BiRefNet failed: {e}')
else:
    print('FAL_KEY not set — skipping BiRefNet')

if cutouts:
    n = len(cutouts)
    fig, axes = plt.subplots(1, n + 1, figsize=((n + 1) * 4, 5))
    axes[0].imshow(imgs[0])
    axes[0].set_title('Original', fontsize=9)
    axes[0].axis('off')
    for ax, (label, cutout) in zip(axes[1:], cutouts.items()):
        display_img = cutout.convert('RGBA')
        bg = Image.new('RGB', display_img.size, (240, 240, 240))
        bg.paste(display_img, mask=display_img.split()[3])
        ax.imshow(bg)
        ax.set_title(f'Cutout — {label}', fontsize=9)
        ax.axis('off')
    plt.suptitle('Step A — Baseline cutouts (not flat-lays)', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('No cutouts produced.')

## Step B — Single-image hero (Gemini, image-conditioned)

**What:** Pass only `imgs[0]` to Gemini with the flat-lay prompt. Tests the minimum viable input: a single worn photo.

**Claim being tested:** One worn photo is enough for Gemini to synthesise a usable flat-lay without hallucinating the garment.

Run both models if you have quota for both.

In [ ]:
recorder = HeroMetricsRecorder()
step_b_results = {}

for model_label, model_id in [('2.5 Flash', IMAGE_GEN_MODEL), ('3.1 Flash', IMAGE_GEN_MODEL_V2)]:
    print(f'[Step B] Generating with {model_label} ({model_id}) — 1 input photo...')
    try:
        with recorder.record(model_label, n_inputs=1, variant='direct') as run:
            hero = generate_hero([imgs[0]], model=model_id)
            run.cost_usd_est = 0.039  # ~$0.039/image per Gemini docs
            run.white_pixel_ratio = compute_white_pixel_ratio(hero)
            run.color_delta_lab = compute_color_delta_lab(imgs[0], hero)
        step_b_results[model_label] = hero
        hero.save(OUT_DIR / f'05_step_b_{model_label.replace(" ", "_")}.png')
        print(f'  ✓  white_ratio={run.white_pixel_ratio:.3f}  color_delta={run.color_delta_lab:.1f}')
    except Exception as e:
        print(f'  ✗  {e}')

if step_b_results:
    cols = len(step_b_results) + 1
    fig, axes = plt.subplots(1, cols, figsize=(cols * 4, 5))
    axes[0].imshow(imgs[0]); axes[0].set_title('Input (worn)', fontsize=9); axes[0].axis('off')
    for ax, (label, hero) in zip(axes[1:], step_b_results.items()):
        ax.imshow(hero); ax.set_title(f'Hero — {label}\n(1 photo)', fontsize=9); ax.axis('off')
    plt.suptitle('Step B — Single-image hero', fontsize=11)
    plt.tight_layout(); plt.show()

## Step C — Multi-image hero (all N photos)

**What:** Pass all loaded photos to Gemini simultaneously. Tests whether more views
improve fidelity to the actual garment (fills in occluded regions, resolves colour ambiguity).

In [ ]:
step_c_results = {}

if len(imgs) < 2:
    print(f'Only {len(imgs)} photo loaded — multi-image test needs ≥2.\nAdd more photos to inputs/worn/ and re-run.')
else:
    for model_label, model_id in [('2.5 Flash', IMAGE_GEN_MODEL), ('3.1 Flash', IMAGE_GEN_MODEL_V2)]:
        print(f'[Step C] Generating with {model_label} — {len(imgs)} input photos...')
        try:
            with recorder.record(model_label, n_inputs=len(imgs), variant='direct') as run:
                hero = generate_hero(imgs, model=model_id)
                run.cost_usd_est = 0.039
                run.white_pixel_ratio = compute_white_pixel_ratio(hero)
                run.color_delta_lab = compute_color_delta_lab(imgs[0], hero)
            step_c_results[model_label] = hero
            hero.save(OUT_DIR / f'05_step_c_{model_label.replace(" ", "_")}.png')
            print(f'  ✓  white_ratio={run.white_pixel_ratio:.3f}  color_delta={run.color_delta_lab:.1f}')
        except Exception as e:
            print(f'  ✗  {e}')

    if step_c_results:
        n_b = len(step_b_results)
        n_c = len(step_c_results)
        cols = 1 + n_b + n_c
        fig, axes = plt.subplots(1, cols, figsize=(cols * 4, 5))
        ax_iter = iter(axes)
        ax = next(ax_iter); ax.imshow(imgs[0]); ax.set_title('Input (worn)', fontsize=9); ax.axis('off')
        for label, hero in step_b_results.items():
            ax = next(ax_iter); ax.imshow(hero); ax.set_title(f'{label}\n1 photo', fontsize=8); ax.axis('off')
        for label, hero in step_c_results.items():
            ax = next(ax_iter); ax.imshow(hero); ax.set_title(f'{label}\n{len(imgs)} photos', fontsize=8); ax.axis('off')
        plt.suptitle('Step B vs Step C — single vs multi photo', fontsize=11)
        plt.tight_layout(); plt.show()

## Step D — Incremental refinement

**What:** Start with 1 photo, add one at a time, render the output after each addition.
Visualises the per-photo gain — does adding photo 3 actually improve on photo 2?

Uses the primary model (`IMAGE_GEN_MODEL`) to keep cost manageable.

In [ ]:
if len(imgs) < 2:
    print('Need ≥2 photos for incremental refinement. Skipping.')
else:
    refinement_heroes = []
    for n in range(1, len(imgs) + 1):
        subset = imgs[:n]
        print(f'[Step D] n={n} photo(s)...')
        try:
            with recorder.record(IMAGE_GEN_MODEL, n_inputs=n, variant='incremental') as run:
                hero = generate_hero(subset, model=IMAGE_GEN_MODEL)
                run.cost_usd_est = 0.039
                run.white_pixel_ratio = compute_white_pixel_ratio(hero)
                run.color_delta_lab = compute_color_delta_lab(imgs[0], hero)
            refinement_heroes.append((n, hero))
            hero.save(OUT_DIR / f'05_step_d_n{n}.png')
            print(f'  ✓  white_ratio={run.white_pixel_ratio:.3f}  color_delta={run.color_delta_lab:.1f}')
        except Exception as e:
            print(f'  ✗  {e}')

    if refinement_heroes:
        cols = len(refinement_heroes)
        fig, axes = plt.subplots(1, cols, figsize=(cols * 4, 5))
        if cols == 1:
            axes = [axes]
        for ax, (n, hero) in zip(axes, refinement_heroes):
            ax.imshow(hero)
            ax.set_title(f'n={n} photo{"s" if n > 1 else ""}', fontsize=9)
            ax.axis('off')
        plt.suptitle('Step D — Incremental refinement (each column adds one photo)', fontsize=11)
        plt.tight_layout(); plt.show()

## Step E — Two-stage variant (cutout prior)

**What:** First remove the background with rembg to get a garment cutout, then feed
the cutout *plus* the original worn photo into Gemini.

**Why:** BridgeDiff (ICPR 2026) argues an explicit flat-structure constraint helps diffusion
models preserve shape. The same intuition applies here — the cutout acts as a structural prior
that constrains Gemini's output geometry.

**Hypothesis:** May outperform direct on complex silhouettes (dresses, draped pieces).
May not help on simpler garments (t-shirts, jeans) where shape is self-evident.

In [ ]:
step_e_results = {}

if not REMBG_AVAILABLE:
    print('rembg not available — skipping two-stage variant. pip install rembg onnxruntime')
else:
    for model_label, model_id in [('2.5 Flash', IMAGE_GEN_MODEL), ('3.1 Flash', IMAGE_GEN_MODEL_V2)]:
        print(f'[Step E] Two-stage ({model_label}) — cutout prior + original...')
        try:
            with recorder.record(model_label, n_inputs=1, variant='cutout-prior') as run:
                hero = generate_hero([imgs[0]], model=model_id, variant='cutout-prior')
                run.cost_usd_est = 0.039
                run.white_pixel_ratio = compute_white_pixel_ratio(hero)
                run.color_delta_lab = compute_color_delta_lab(imgs[0], hero)
            step_e_results[model_label] = hero
            hero.save(OUT_DIR / f'05_step_e_{model_label.replace(" ", "_")}.png')
            print(f'  ✓  white_ratio={run.white_pixel_ratio:.3f}  color_delta={run.color_delta_lab:.1f}')
        except Exception as e:
            print(f'  ✗  {e}')

    if step_e_results:
        cols = 1 + len(step_e_results)
        fig, axes = plt.subplots(1, cols, figsize=(cols * 4, 5))
        axes[0].imshow(imgs[0]); axes[0].set_title('Input (worn)', fontsize=9); axes[0].axis('off')
        for ax, (label, hero) in zip(axes[1:], step_e_results.items()):
            ax.imshow(hero)
            ax.set_title(f'Two-stage — {label}', fontsize=9)
            ax.axis('off')
        plt.suptitle('Step E — Two-stage variant (cutout prior)', fontsize=11)
        plt.tight_layout(); plt.show()

## Step F — Metrics table

One row per (provider, n_inputs, variant) combination. Fill in `eyeball_quality` manually after inspecting outputs.

**Columns:**
- **Color Δ LAB** — mean CIELAB distance between input garment pixels and hero output pixels. Lower = more faithful colour.
- **BG white ratio** — fraction of near-white pixels in the output. Higher = cleaner background removal.
- **Eyeball quality (1–5)** — set manually by running the last cell.

In [ ]:
df = recorder.to_dataframe()

# Fill eyeball scores manually here after inspecting outputs/05/
# Example: df.loc[0, 'Eyeball quality (1-5)'] = 4

print(df.to_string(index=False))
df

## Decision note

> **Fill in after running.** Copy the winning row from the metrics table and answer the three questions below.

**Winner:** `<provider> / n_inputs=<N> / variant=<direct|cutout-prior>`

**Is one worn photo enough?**
- [ ] Yes — Step B single-image hero is production-viable.
- [ ] No — Need ≥2 photos; the improvement in Step D was worth it.
- [ ] Marginal — Single photo works for simple silhouettes; multi helps for complex ones.

**Does the cutout prior (Step E) beat direct?**
- [ ] Yes — ships as default `variant='cutout-prior'` in `src/ai/heroImage.ts`
- [ ] No — direct is simpler and equally good
- [ ] Conditionally — use cutout-prior only for `category in ['dresses', 'outerwear']`

**Ship recommendation for `src/ai/heroImage.ts`:**
```
model:   <winning model ID>
variant: direct | cutout-prior
n_inputs required: 1 | 2+
```

**Scoring rubric for production use:**
Flag a generation for manual review if `white_pixel_ratio < 0.3` (dirty background) or `color_delta_lab > 25` (colour drift > perceptible threshold).